In [ ]:
# !pip install prophet

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 722.4 kB/s eta 0:00:17
   ----- ---------------------------------- 1.6/12.1 MB 2.3 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.1 MB 3.2 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.1 MB 3.7 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 4.1 MB/s eta 0:00:02
   --------------------- ------------------ 6.6/12.1 MB 4.4 MB/s eta 0:00:02
   ------------------------- -------------- 7.9/12.1 MB 4.7 MB/s eta 0:00:01
   ------------------------------ --------- 9.2/12.1 MB 4.8 MB/s eta 0:00:01
   -------------------------------- ------- 10.0/12.1 MB 4.8 MB/s eta 0:00:01
   ----------------------------------- ---- 10.7/12.1 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.1 MB 4.8 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 기획서 기반 단기 팩터 계산 (Rolling Window)
# ---------------------------------------------------------
def calculate_short_term_factors(df):
    """
    기획서 Page 3~4에 정의된 단기 팩터(S1~S5) 계산
    입력 df: MultiIndex (date, ticker) 또는 date, ticker 컬럼이 있는 데이터프레임
    필수 컬럼: Close, Open, TradingValue(거래대금), Marcap(시가총액), NetBuy_Foreign(외국인순매수)
    """
    # 계산 편의를 위해 종목별로 그룹화
    grouped = df.groupby('ticker')
    
    # 결과 담을 DataFrame 복사
    factors = df.copy()
    
    # -----------------------------------------------------
    # S1_MOM_20D: 최근 20일 모멘텀
    # 정의: close_t / close_{t-20} - 1
    # -----------------------------------------------------
    factors['S1_MOM_20D'] = grouped['Close'].transform(lambda x: x.pct_change(periods=20))
    
    # -----------------------------------------------------
    # S2_VOL_RATIO_20D: 거래대금 급증 여부
    # 정의: 최근 20일 평균 거래대금 / 최근 100일 평균 거래대금
    # -----------------------------------------------------
    # min_periods=1로 설정하여 초기 데이터 부족 시에도 계산되게 하거나, 엄격하게 하려면 100
    ma_20_vol = grouped['TradingValue'].transform(lambda x: x.rolling(window=20).mean())
    ma_100_vol = grouped['TradingValue'].transform(lambda x: x.rolling(window=100).mean())
    factors['S2_VOL_RATIO_20D'] = ma_20_vol / ma_100_vol

    # -----------------------------------------------------
    # S3_INTRADAY_RET_20D: 장중 강세 강도
    # 정의: 최근 20일 동안 (종가/시가 - 1)의 평균
    # -----------------------------------------------------
    # 먼저 당일 장중 수익률 계산
    intraday_ret = (df['Close'] / df['Open']) - 1
    factors['intraday_ret'] = intraday_ret
    # 20일 이동평균
    factors['S3_INTRADAY_RET_20D'] = factors.groupby('ticker')['intraday_ret'].transform(
        lambda x: x.rolling(window=20).mean()
    )
    
    # S4_NEWS_SENT_5D는 원천 데이터 부재로 생략 (기획서 Page 4 내용 반영)
    factors['S4_NEWS_SENT_5D'] = np.nan 

    # -----------------------------------------------------
    # S5_FOREIGN_FLOW_20D: 외국인 수급 강도
    # 정의: 최근 20일 외국인 순매수 누적 / 시가총액
    # -----------------------------------------------------
    # 20일 누적 합계
    cum_foreign_20 = grouped['NetBuy_Foreign'].transform(lambda x: x.rolling(window=20).sum())
    factors['S5_FOREIGN_FLOW_20D'] = cum_foreign_20 / df['Marcap']
    
    # 불필요한 임시 컬럼 제거
    factors.drop(columns=['intraday_ret'], inplace=True, errors='ignore')
    
    return factors

# ---------------------------------------------------------
# 2. Prophet 기반 단기 예측 (Rolling Forecast)
# ---------------------------------------------------------
def get_prophet_trend_score(ticker_df, forecast_days=20):
    """
    특정 종목의 시계열 데이터를 받아 Prophet으로 미래를 예측하고
    '예측 수익률(Trend)'을 반환하는 함수.
    
    실전에서는 매일 모든 종목에 돌리기 무거우므로, 
    월 1회 리밸런싱 시점에만 호출하는 것을 권장합니다.
    """
    # 1. Prophet 형식에 맞게 데이터 변환 (Date -> ds, Close -> y)
    df_prophet = ticker_df.reset_index()[['date', 'Close']].rename(columns={'date': 'ds', 'Close': 'y'})
    
    # 데이터가 너무 적으면 예측 불가 (예: 6개월 미만)
    if len(df_prophet) < 120:
        return np.nan
        
    # 2. 모델 생성 및 학습
    # changepoint_prior_scale: 추세 변화 유연성 조절 (기본 0.05, 주가는 변동성이 크므로 약간 높임)
    model = Prophet(daily_seasonality=True, changepoint_prior_scale=0.1)
    model.fit(df_prophet)
    
    # 3. 미래 데이터프레임 생성
    future = model.make_future_dataframe(periods=forecast_days)
    
    # 4. 예측
    forecast = model.predict(future)
    
    # 5. 스코어 산출: (미래 마지막 날 예측가 / 현재 예측가) - 1
    # yhat: 트렌드와 계절성이 반영된 예측값
    current_pred = forecast['yhat'].iloc[-forecast_days-1] # 오늘(학습 마지막 날)의 fitting 값
    future_pred = forecast['yhat'].iloc[-1]               # 미래 마지막 날 값
    
    predicted_return = (future_pred / current_pred) - 1
    
    return predicted_return

# ---------------------------------------------------------
# 3. 실행 예시 (가상의 데이터 생성)
# ---------------------------------------------------------
if __name__ == "__main__":
    # 가상의 데이터 생성 (2개 종목, 200일 치)
    dates = pd.date_range(start='2024-01-01', periods=200, freq='B')
    ticker_list = ['005930', '000660'] # 삼성전자, SK하이닉스 예시
    
    data = []
    for ticker in ticker_list:
        for d in dates:
            # 랜덤 데이터 생성
            close = np.random.uniform(60000, 80000)
            open_ = close * np.random.uniform(0.98, 1.02)
            vol = np.random.uniform(100000, 1000000)
            data.append({
                'date': d,
                'ticker': ticker,
                'Close': close,
                'Open': open_,
                'TradingValue': vol * close, # 거래대금
                'Marcap': close * 50000000,  # 시가총액 (가정)
                'NetBuy_Foreign': np.random.uniform(-100000000, 100000000) # 외국인 순매수 금액
            })
            
    df = pd.DataFrame(data)
    
    # 1) 기본 Rolling 팩터 계산
    print(">>> Rolling 팩터 계산 중...")
    df_factors = calculate_short_term_factors(df)
    
    print(df_factors[['date', 'ticker', 'S1_MOM_20D', 'S2_VOL_RATIO_20D', 'S5_FOREIGN_FLOW_20D']].tail())
    
    # 2) Prophet 예측 적용 (최신 날짜 기준)
    # 실제 리밸런싱 날짜(예: 2024-10-04)에만 수행한다고 가정
    target_date = dates[-1] 
    print(f"\n>>> Prophet 모델 예측 수행 중 (기준일: {target_date.date()})...")
    
    prophet_scores = []
    
    for ticker in ticker_list:
        # 해당 종목의 과거 데이터 추출
        sub_df = df[df['ticker'] == ticker].sort_values('date')
        
        # Prophet 예측 수행
        score = get_prophet_trend_score(sub_df, forecast_days=20)
        
        prophet_scores.append({
            'date': target_date,
            'ticker': ticker,
            'S6_PROPHET_PRED': score
        })
        print(f"[{ticker}] 향후 20일 예측 수익률: {score:.4f}")

    # 결과 합치기
    df_prophet = pd.DataFrame(prophet_scores)
    
    # 최종적으로 사용할 팩터 테이블
    # (실제 프로젝트에서는 이를 날짜별로 merge 하여 사용)
    print("\n>>> 최종 단기 팩터 스코어 (일부):")
    print(df_prophet)

Importing plotly failed. Interactive plots will not work.


>>> Rolling 팩터 계산 중...
          date  ticker  S1_MOM_20D  S2_VOL_RATIO_20D  S5_FOREIGN_FLOW_20D
395 2024-09-30  000660    0.089417          1.117189            -0.000077
396 2024-10-01  000660   -0.030993          1.112568            -0.000063
397 2024-10-02  000660   -0.110059          1.068220            -0.000110
398 2024-10-03  000660   -0.151899          1.017550            -0.000116
399 2024-10-04  000660   -0.125595          1.007681            -0.000130

>>> Prophet 모델 예측 수행 중 (기준일: 2024-10-04)...


11:13:50 - cmdstanpy - INFO - Chain [1] start processing
11:13:51 - cmdstanpy - INFO - Chain [1] done processing
11:13:51 - cmdstanpy - INFO - Chain [1] start processing


[005930] 향후 20일 예측 수익률: -0.0207


11:13:51 - cmdstanpy - INFO - Chain [1] done processing


[000660] 향후 20일 예측 수익률: 0.0049

>>> 최종 단기 팩터 스코어 (일부):
        date  ticker  S6_PROPHET_PRED
0 2024-10-04  005930        -0.020684
1 2024-10-04  000660         0.004898


In [6]:
import pandas as pd
import numpy as np
from prophet import Prophet

# ---------------------------------------------------------
# 1. 더미 데이터 생성 (이전과 동일)
# ---------------------------------------------------------
dates = pd.date_range(start='2024-01-01', periods=200, freq='B')
# 예시를 위해 5개 종목으로 늘려서 순위를 비교해봅시다
ticker_list = ['005930', '000660', '035420', '005380', '051910'] 
# (삼성전자, SK하이닉스, NAVER, 현대차, LG화학)

data = []
np.random.seed(42) # 결과 고정을 위해 시드 설정

for ticker in ticker_list:
    # 종목마다 조금씩 다른 추세를 줌
    trend = np.random.uniform(-0.001, 0.001) 
    price = 70000
    
    for d in dates:
        price = price * (1 + trend + np.random.normal(0, 0.02)) # 랜덤 변동
        data.append({
            'date': d,
            'ticker': ticker,
            'Close': abs(price),
            'Open': abs(price * np.random.uniform(0.99, 1.01)),
            'TradingValue': np.random.uniform(1e9, 1e10), # 거래대금
            'Marcap': abs(price) * 1e6,  # 시가총액
            'NetBuy_Foreign': np.random.uniform(-1e8, 1e8) # 외국인 수급
        })

df = pd.DataFrame(data)

# ---------------------------------------------------------
# 2. 팩터 계산 로직 (기획서 반영)
# ---------------------------------------------------------
def calculate_short_term_factors(df):
    grouped = df.groupby('ticker')
    factors = df.copy()
    
    # S1: 모멘텀 (수익률)
    factors['S1_MOM_20D'] = grouped['Close'].transform(lambda x: x.pct_change(periods=20))
    
    # S2: 거래대금 급증 (최근20일 / 과거100일)
    ma_20 = grouped['TradingValue'].transform(lambda x: x.rolling(20).mean())
    ma_100 = grouped['TradingValue'].transform(lambda x: x.rolling(100).mean())
    factors['S2_VOL_RATIO'] = ma_20 / ma_100
    
    # S3: 장중 강세 (시가 대비 종가)
    factors['intraday'] = (df['Close'] / df['Open']) - 1
    factors['S3_INTRADAY'] = factors.groupby('ticker')['intraday'].transform(lambda x: x.rolling(20).mean())
    
    # S5: 외국인 수급 (시총 대비 순매수 비중)
    factors['S5_FOREIGN'] = grouped['NetBuy_Foreign'].transform(lambda x: x.rolling(20).sum()) / df['Marcap']
    
    return factors.dropna()

# ---------------------------------------------------------
# 3. Prophet 예측 로직 (조용하게 실행)
# ---------------------------------------------------------
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING) # 로그 끄기

def get_prophet_score(sub_df):
    if len(sub_df) < 60: return np.nan
    
    df_p = sub_df.reset_index()[['date', 'Close']].rename(columns={'date':'ds', 'Close':'y'})
    
    m = Prophet(daily_seasonality=True, changepoint_prior_scale=0.1)
    m.fit(df_p)
    
    future = m.make_future_dataframe(periods=20)
    forecast = m.predict(future)
    
    # 현재가 대비 미래 예측가 수익률
    curr = forecast['yhat'].iloc[-21]
    fut = forecast['yhat'].iloc[-1]
    return (fut / curr) - 1

# ---------------------------------------------------------
# 4. [핵심] 점수 정규화 및 종합 (0~100점 변환)
# ---------------------------------------------------------
def normalize_and_score(daily_df):
    """
    하루치(cross-section) 데이터를 받아서 0~100점으로 변환
    """
    scored = daily_df.copy()
    factor_cols = ['S1_MOM_20D', 'S2_VOL_RATIO', 'S3_INTRADAY', 'S5_FOREIGN', 'S6_PROPHET']
    
    for col in factor_cols:
        if col in scored.columns:
            # 1. 순위 매기기 (pct=True로 0~1 사이 값으로 변환)
            # 기획서: higher_is_better=True (높을수록 좋음) -> 그대로 사용
            rank = scored[col].rank(pct=True, ascending=True)
            
            # 2. 100점 만점으로 변환
            scored[f'{col}_SCORE'] = rank * 100
            
    # 종합 점수 (평균)
    score_cols = [c for c in scored.columns if '_SCORE' in c]
    scored['FINAL_SHORT_SCORE'] = scored[score_cols].mean(axis=1)
    
    return scored

# =========================================================
# 실행
# =========================================================

# 1. 기본 팩터 계산
print("1. 기본 팩터 계산 중...")
df_factors = calculate_short_term_factors(df)

# 2. 특정 날짜(오늘) 기준으로 자르기
target_date = dates[-1]
today_df = df_factors[df_factors['date'] == target_date].copy()

# 3. Prophet 예측 추가
print("2. AI 모델 예측 중 (시간이 조금 걸립니다)...")
today_df['S6_PROPHET'] = today_df.apply(
    lambda row: get_prophet_score(df[df['ticker'] == row['ticker']]), axis=1
)

# 4. 점수 산출
print("3. 최종 점수 채점 중...")
final_df = normalize_and_score(today_df)

# =========================================================
# 결과 출력
# =========================================================
print("\n" + "="*50)
print(f"[{target_date.date()}] AI Advisor 단기 트랙 추천 순위")
print("="*50)

# 보기 좋게 컬럼 선택 및 정렬
display_cols = ['ticker', 'FINAL_SHORT_SCORE', 'S1_MOM_20D_SCORE', 'S6_PROPHET_SCORE']
result = final_df.sort_values('FINAL_SHORT_SCORE', ascending=False)[display_cols]

# 소수점 정리
print(result.round(1).to_string(index=False))

print("\n>>> 해석:")
print(" - FINAL_SHORT_SCORE: 100점에 가까울수록 '지금 사기 좋은' 종목입니다.")
print(" - S6_PROPHET_SCORE: AI가 예측한 미래 추세 점수입니다.")

1. 기본 팩터 계산 중...
2. AI 모델 예측 중 (시간이 조금 걸립니다)...
3. 최종 점수 채점 중...

[2024-10-04] AI Advisor 단기 트랙 추천 순위
ticker  FINAL_SHORT_SCORE  S1_MOM_20D_SCORE  S6_PROPHET_SCORE
005930               72.0              80.0              80.0
005380               72.0              60.0             100.0
051910               56.0              40.0              60.0
035420               52.0             100.0              40.0
000660               48.0              20.0              20.0

>>> 해석:
 - FINAL_SHORT_SCORE: 100점에 가까울수록 '지금 사기 좋은' 종목입니다.
 - S6_PROPHET_SCORE: AI가 예측한 미래 추세 점수입니다.
